Structured streaming project

In [0]:
import json
import time
import random
from datetime import datetime

# 1. Create Unity Catalog Namespace
spark.sql("CREATE CATALOG IF NOT EXISTS main;")
spark.sql("CREATE SCHEMA IF NOT EXISTS main.uber_streaming_catalog;")

# 2. Create a Managed/External Unity Catalog Volume (replaces legacy DBFS)
spark.sql("CREATE VOLUME IF NOT EXISTS main.uber_streaming_catalog.uber_volume;")

# Define UC Volume paths
volume_base = "/Volumes/main/uber_streaming_catalog/uber_volume"
source_path = f"{volume_base}/raw_source"
checkpoint_path = f"{volume_base}/checkpoints/autoloader"

# Note: Volumes support standard Python file I/O operations directly
import os
os.makedirs(source_path, exist_ok=True)
os.makedirs(checkpoint_path, exist_ok=True)

# 3. Simulate a raw JSON ride event inside the Unity Catalog Volume
sample_ride = {
    "ride_id": f"ride_{random.randint(1000, 9999)}",
    "rider_id": f"user_{random.randint(100, 500)}",
    "pickup_location": "Indiranagar",
    "dropoff_location": "Koramangala",
    "fare_amount": round(random.uniform(150.0, 800.0), 2),
    "distance_km": round(random.uniform(2.0, 18.0), 2),
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

file_path = f"{source_path}/ride_event_{int(time.time())}.json"
with open(file_path, "w") as f:
    json.dump(sample_ride, f)

print(f"Sample file successfully written to Unity Catalog Volume at: {file_path}")

In [0]:
from pyspark.sql.functions import col, window, expr, when
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

# Define explicit schema for incoming Uber JSON payload
uber_schema = StructType([
    StructField("ride_id", StringType(), True),
    StructField("rider_id", StringType(), True),
    StructField("pickup_location", StringType(), True),
    StructField("dropoff_location", StringType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("distance_km", DoubleType(), True),
    StructField("timestamp", StringType(), True)
])

# 1. Incremental Ingestion via Auto Loader reading from UC Volume
raw_stream_df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema") \
    .schema(uber_schema) \
    .load(source_path)

# 2. Interview-grade PySpark Transformations:
# - Filter invalid records
# - Cast timestamps for windowing
# - Conditional logic: categorize trip lengths using when().otherwise()
# - Watermarking to manage state and late data in streaming
transformed_df = raw_stream_df \
    .withColumn("event_time", col("timestamp").cast(TimestampType())) \
    .filter(col("ride_id").isNotNull() & (col("fare_amount") > 0)) \
    .withColumn("distance_category", 
        when(col("distance_km") < 5.0, "Short Trip")
        .when((col("distance_km") >= 5.0) & (col("distance_km") <= 15.0), "Medium Trip")
        .otherwise("Long Trip")
    ) \
    .withWatermark("event_time", "10 minutes")

# 3. Window Function Aggregation (10-minute Tumbling Windows)
windowed_agg_df = transformed_df \
    .groupBy(
        window(col("event_time"), "10 minutes"),
        col("pickup_location"),
        col("distance_category")
    ) \
    .agg(
        expr("count(ride_id) as total_rides"),
        expr("avg(fare_amount) as avg_fare"),
        expr("sum(fare_amount) as total_revenue")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("pickup_location"),
        col("distance_category"),
        col("total_rides"),
        col("avg_fare"),
        col("total_revenue")
    )

# 4. Write stream directly to a Unity Catalog Managed Table using 3-level namespace
# NOTE: Serverless compute does NOT support the default ProcessingTime (continuous) trigger.
#       Use trigger(availableNow=True) which processes all available data in one batch,
#       then stops — the correct approach for interactive Serverless streaming.
# NOTE: Delta sink does not support outputMode("update") directly via .table().
#       Use foreachBatch to write aggregated updates to the Delta table.
def write_to_delta(batch_df, batch_id):
    batch_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("main.uber_streaming_catalog.uber_windowed_metrics")

query = windowed_agg_df.writeStream \
    .outputMode("update") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", f"{checkpoint_path}/query") \
    .foreachBatch(write_to_delta) \
    .start()

# Wait for the AvailableNow query to finish processing all available data
query.awaitTermination()

print("Streaming pipeline successfully writing to Unity Catalog table: main.uber_streaming_catalog.uber_windowed_metrics")

In [0]:
display(spark.sql("SELECT * FROM main.uber_streaming_catalog.uber_windowed_metrics ORDER BY window_end DESC"))